# RT-DETR on DocLayNet

## hardware check


In [ ]:
!nvidia-smi

## Install


In [ ]:
!pip install -q ultralytics==8.3.40 "datasets<4.0.0"


## Pull the repo

In [ ]:
REPO_URL = "https://github.com/RISHIVELS/doc_layout_detection.git"

# %cd into a directory this cell is about to rm -rf breaks the shell's
# own working directory on the *next* run of this cell (getcwd fails
# because the path it thinks it is standing in no longer exists). So I
# step out to a stable parent directory before deleting anything.
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1


## Build the dataset

~3.8GB first run.

In [ ]:
!python scripts/prepare_dataset.py --out /kaggle/working/data/doclaynet --verify 12

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

checks = sorted(Path("/kaggle/working/data/doclaynet/label_check").glob("*.png"))
print(f"{len(checks)} pages to check\n")

for path in checks[:4]:
    display(Image.open(path).resize((620, 620)))

## Train


In [ ]:
!python scripts/train.py \
    --data /kaggle/working/data/doclaynet/doclaynet.yaml \
    --epochs 30 \
    --batch 8 \
    --imgsz 640 \
    --device 0 \
    --name rtdetr_doclaynet

## Evaluate

Per-class AP, per-category mAP, query saturation. Category breakdown
matters most - aggregate mAP can't tell you if it learned structure
or just financial reports.

In [ ]:
!python scripts/evaluate.py \
    --weights runs/detect/rtdetr_doclaynet/weights/best.pt \
    --data /kaggle/working/data/doclaynet/doclaynet.yaml \
    --out reports

## Mine failure cases

Ranks worst predictions, renders them against ground truth. Memo's
failure cases come from here, not guessing.

In [ ]:
!python scripts/mine_failures.py \
    --weights runs/detect/rtdetr_doclaynet/weights/best.pt \
    --data /kaggle/working/data/doclaynet \
    --out reports/failures \
    --top 25

## Outputs

In [ ]:
import shutil
from pathlib import Path

run = Path("runs/detect/rtdetr_doclaynet")
out = Path("/kaggle/working/submission")
out.mkdir(exist_ok=True)

shutil.copy(run / "weights/best.pt", out / "best.pt")
shutil.copy(run / "run_metadata.json", out / "run_metadata.json")
if Path("reports").exists():
    shutil.copytree("reports", out / "reports", dirs_exist_ok=True)

for path in sorted(out.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:8.1f} MB  {path.relative_to(out)}")